# Complementary Categories

Which categories get bought together. `also_buy` is the signal: for each item
Amazon lists the asins shown as "frequently bought together", so mapping both
ends of every edge to its category turns a product-level co-purchase list into
something a category-by-category complementarity matrix aggregates from.

Two sources, and the split between them matters:

| Source | Grain | Role |
| --- | --- | --- |
| `df_features.pkl` | one row per `asin`, schema categories only | the **source** side of each edge, carrying the extracted features |
| `meta_Home_and_Kitchen_filtered.csv` | one row per `asin`, whole catalogue | the **target** side lookup |

`run_feature_extraction` calls `filter_by_cat_3`, so `df_features.pkl` holds
only items whose `cat_3` is one of the 69 categories in
`master_metadata.json`. `also_buy` points anywhere — other categories, and
outside Home & Kitchen entirely — so target categories are looked up in the
CSV, which was never filtered. Edges whose target is in neither table are kept
with their categories marked `Not in catalogue`: how much of `also_buy` leaves
the catalogue is a finding, not something to drop on the floor.

`cat_4` is folded through `category_taxonomy.json` here, the same whitelist
`ttn.ipynb` §3 applies — the pipeline does not produce `cat_4_clean`. `cat_2`
and `cat_3` have no cleaned variant and are used as parsed.

**`also_buy` must be in the pickle.** It only reaches `df_features.pkl` if the
extraction ran against a CSV that already carried the column — that is, one
rebuilt by `data/variable_selection.ipynb` after `also_buy` was added to
`fields_to_keep`. §1 checks and says so if not.

In [51]:
import ast
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Notebooks run from notebooks/, so the project root is one level up.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
FEATURES_PATH = DATA_DIR / "df_features.pkl"
CATALOGUE_PATH = DATA_DIR / "meta_Home_and_Kitchen_filtered.csv"
TAXONOMY_PATH = DATA_DIR / "category_taxonomy.json"
CATEGORY_MAP_PATH = DATA_DIR / "complementary_category_map.csv"

# Show every column/variable when displaying a dataframe
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 50)
# Turn off scientific notation (e.g. 2.447268e+06 -> 2447268.00)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Load the extracted features

`df_features.pkl` is the raw metadata with the extraction results already
joined on it, so loading it is what connects the categories and `also_buy` to
the features. Everything else in this notebook works off `asin`.

In [52]:
df_features = pd.read_pickle(FEATURES_PATH)
print(f"df_features: {df_features.shape[0]:,} rows x {df_features.shape[1]} cols")

if "also_buy" not in df_features.columns:
    raise KeyError(
        "df_features.pkl has no 'also_buy' column. The extraction ran against a "
        "CSV built before 'also_buy' was added to fields_to_keep in "
        "data/variable_selection.ipynb. Rebuild the CSV there, then re-run "
        "feature_extraction_workflow/extract_features.ipynb."
    )

print(f"\nunique asins   : {df_features['asin'].nunique():,}")
print(f"cat_3 values   : {df_features['cat_3'].nunique():,} (the schema categories)")
print(f"rows with features: {(df_features['extracted_features'].apply(len) > 0).sum():,}")
df_features[["asin", "cat_2", "cat_3", "cat_4", "also_buy", "extracted_features"]].head(3)

df_features: 1,134,566 rows x 92 cols

unique asins   : 1,134,566
cat_3 values   : 69 (the schema categories)
rows with features: 1,112,629


,asin,cat_2,cat_3,cat_4,also_buy,extracted_features
0,0001487795,Kitchen & Dining,Dining & Entertaining,Dinnerware,"['B0001XR2F2', 'B01LY51HUN', 'B07CXZ9C5B', '03...",{'Color': 'red'}
1,0002020300,Home Dcor,Candles & Holders,Candles,[],{}
2,0006564224,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,[],"{'Product_Type': 'cup', 'Capacity_Volume': '16..."


## 2. Clean `cat_4` against the reviewed taxonomy

`category_taxonomy.json` is a whitelist of the `(cat_3, cat_4)` pairs that are
real categories rather than product bullets that leaked into the category path
— `'Imported'`, `'10" high'`, `'measures 25x25x14cm'` and 550-odd others like
them. A value is kept only if it is valid **under its own parent**, since the
same label can be real in one branch and junk in another; everything else
becomes `<cat_3>_Other`.

The same function is applied to the catalogue lookup in §4, so both ends of an
edge are cleaned identically.

In [53]:
with open(TAXONOMY_PATH) as f:
    taxonomy = json.load(f)

VALID_PAIRS = {
    (cat_3, value)
    for cat_2 in taxonomy
    for cat_3, values in taxonomy[cat_2].items()
    for value in values
}
MISSING = "Missing"
OTHER_SUFFIX = "_Other"

print(f"taxonomy: {len(taxonomy)} cat_2 | "
      f"{sum(len(v) for v in taxonomy.values())} cat_3 | "
      f"{len(VALID_PAIRS):,} valid (cat_3, cat_4) pairs")


def fold_cat_4(cat_3: pd.Series, cat_4: pd.Series) -> pd.Series:
    """Keep cat_4 where the (cat_3, cat_4) pair survived review, else <cat_3>_Other."""
    c3 = cat_3.astype(str)
    c4 = cat_4.fillna(MISSING).astype(str)
    keep = pd.Series(list(zip(c3, c4)), index=c3.index).isin(VALID_PAIRS)
    return pd.Series(np.where(keep, c4, c3 + OTHER_SUFFIX), index=c3.index)


df_features["cat_4_clean"] = fold_cat_4(df_features["cat_3"], df_features["cat_4"])

n_folded = int((df_features["cat_4_clean"] != df_features["cat_4"].fillna(MISSING)).sum())
print(f"\ndistinct cat_4: {df_features['cat_4'].nunique(dropna=False):,} -> "
      f"{df_features['cat_4_clean'].nunique():,}")
print(f"items folded into '<cat_3>{OTHER_SUFFIX}': {n_folded:,} "
      f"({n_folded / len(df_features):.2%})")

taxonomy: 7 cat_2 | 69 cat_3 | 521 valid (cat_3, cat_4) pairs

distinct cat_4: 921 -> 451
items folded into '<cat_3>_Other': 1,099 (0.10%)


## 3. The base table — one row per product

`asin`, its cleaned category path, its image url and its `also_buy` list. The
raw `also_buy` column is a stringified list (`"['B0001XR2F2', ...]"`), so it is
parsed back to a real list here; anything unparseable becomes an empty list
rather than an error, and the count below shows whether that is happening at
any scale.

`imageURLHighRes` and `imageURL` are stringified lists too — several shots of
the same product, frequently `[]`. Only the first url is kept, high-res
preferred, and it is pulled with a regex rather than a per-row parse because
the same helper runs over the whole 2 GB catalogue in §4. Items with no image
keep a null: roughly a third of the catalogue has one, so dropping the rest
would silently throw away most of the co-purchase edges.

In [54]:
IMAGE_URL_RE = r"[\'\"](https?://[^\'\"]+)[\'\"]"


def parse_asin_list(value) -> list:
    """Stringified list -> list of asins. Anything unparseable -> []."""
    if isinstance(value, list):
        return value
    if not isinstance(value, str) or not value.strip():
        return []
    try:
        parsed = ast.literal_eval(value)
    except (ValueError, SyntaxError):
        return []
    return parsed if isinstance(parsed, list) else []


def first_image_url(high_res: pd.Series, fallback: pd.Series) -> pd.Series:
    """First url of imageURLHighRes, else of imageURL. No image at all -> NaN."""
    urls = high_res.astype(str).str.extract(IMAGE_URL_RE, expand=False)
    return urls.fillna(fallback.astype(str).str.extract(IMAGE_URL_RE, expand=False))


df_base = df_features[["asin", "cat_2", "cat_3", "cat_4_clean"]].copy()
df_base["image_url"] = first_image_url(
    df_features["imageURLHighRes"], df_features["imageURL"]
)
df_base["also_buy"] = df_features["also_buy"].apply(parse_asin_list)
df_base["also_buy_n"] = df_base["also_buy"].str.len()

with_edges = int((df_base["also_buy_n"] > 0).sum())
with_image = int(df_base["image_url"].notna().sum())
print(f"products                 : {len(df_base):,}")
print(f"with a non-empty also_buy: {with_edges:,} ({with_edges / len(df_base):.1%})")
print(f"with an image url        : {with_image:,} ({with_image / len(df_base):.1%})")
print(f"co-purchase references   : {df_base['also_buy_n'].sum():,}")
print(f"per product, mean / max  : {df_base['also_buy_n'].mean():.1f} / "
      f"{df_base['also_buy_n'].max():,}")
print(f"median among non-empty   : "
      f"{df_base.loc[df_base['also_buy_n'] > 0, 'also_buy_n'].median():.0f}")

# extract_features.ipynb loads the CSV with .drop_duplicates(), so this should
# be 0. If it isn't, every edge from a repeated asin is counted twice in §5.
n_dupe_src = int(df_base["asin"].duplicated().sum())
if n_dupe_src:
    print()
    print(f"!! {n_dupe_src:,} duplicate source asins — §5 edges will double-count")

df_base[df_base["also_buy_n"] > 0].head(5)

products                 : 1,134,566
with a non-empty also_buy: 154,769 (13.6%)
with an image url        : 558,717 (49.2%)
co-purchase references   : 3,098,540
per product, mean / max  : 2.7 / 100
median among non-empty   : 9


,asin,cat_2,cat_3,cat_4_clean,image_url,also_buy,also_buy_n
0,0001487795,Kitchen & Dining,Dining & Entertaining,Dinnerware,NaN,"[B0001XR2F2, B01LY51HUN, B07CXZ9C5B, 0310258952]",4
7,0439903491,Wall Art,Posters & Prints,Missing,https://images-na.ssl-images-amazon.com/images...,[0439900581],1
8,0456680012,Bedding,Kids' Bedding,Duvet Covers & Sets,https://images-na.ssl-images-amazon.com/images...,"[B00I8TCB02, B00K5B0PCC, B00P8BOYNU, B01AB1CFA...",13
9,0470902884,Wall Art,Posters & Prints,Missing,NaN,[0470559721],1
22,0752873016,Wall Art,Posters & Prints,Missing,NaN,"[1510100466, 0752860682, 1444000276, 144401167...",96


## 4. Category lookup over the whole catalogue

`also_buy` targets are mostly *not* in `df_features` — that table stops at the
69 schema categories, while a co-purchase edge can point at any item. The
filtered CSV was never narrowed that way, so it is the lookup. Only `asin`,
`category` and the two image columns are read; the rest of the 2 GB stays on
disk.

Items whose `cat_3` is outside the taxonomy get `<cat_3>_Other` from
`fold_cat_4`, which is the honest answer: unreviewed, so not trusted as a
leaf.

About a quarter of the CSV is exact full-row duplicates (60k sampled rows held
44,852 asins), which is why the dedupe count below is large — the same rows
`extract_features.ipynb` drops on load with `.drop_duplicates()`. They are byte
-identical repeats, so `keep="first"` discards no information.

The category path is parsed here rather than with
`feature_extraction_workflow.ensure_cat_columns`, which does the same thing:
importing that package pulls in `nltk` at module load, and the `RecSystem
(venv)` kernel does not have it. Nothing else in this notebook needs the
package, so it stays independent of that.

In [55]:
def parse_category_levels(series: pd.Series, n_levels: int = 4) -> pd.DataFrame:
    """Stringified category path -> cat_1..cat_n columns (same rule as the pipeline)."""
    def as_list(value):
        if isinstance(value, list):
            return value
        if not isinstance(value, str) or not value.strip():
            return []
        try:
            parsed = ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return []
        return parsed if isinstance(parsed, list) else []

    paths = series.apply(as_list)
    return pd.DataFrame(
        {f"cat_{i + 1}": paths.apply(lambda p, i=i: p[i] if len(p) > i else None)
         for i in range(n_levels)},
        index=series.index,
    )


cat_lookup = pd.read_csv(
    CATALOGUE_PATH,
    usecols=["asin", "category", "imageURL", "imageURLHighRes"],
    low_memory=False,
)
print(f"catalogue rows: {len(cat_lookup):,}")

cat_lookup = pd.concat(
    [
        cat_lookup[["asin"]],
        parse_category_levels(cat_lookup["category"]),
        first_image_url(
            cat_lookup["imageURLHighRes"], cat_lookup["imageURL"]
        ).rename("image_url"),
    ],
    axis=1,
)
cat_lookup["cat_4_clean"] = fold_cat_4(cat_lookup["cat_3"], cat_lookup["cat_4"])
cat_lookup = cat_lookup[["asin", "cat_2", "cat_3", "cat_4_clean", "image_url"]]

n_dupes = int(cat_lookup["asin"].duplicated().sum())
cat_lookup = cat_lookup.drop_duplicates("asin", keep="first")
with_image = int(cat_lookup["image_url"].notna().sum())
print(f"duplicate asins dropped: {n_dupes:,}")
print(f"lookup covers: {len(cat_lookup):,} asins "
      f"({len(cat_lookup) / len(df_base):.1f}x the {len(df_base):,} in df_features)")
print(f"with an image url: {with_image:,} ({with_image / len(cat_lookup):.1%})")
cat_lookup.head(3)

catalogue rows: 1,300,540
duplicate asins dropped: 15,148
lookup covers: 1,285,392 asins (1.1x the 1,134,566 in df_features)
with an image url: 637,823 (49.6%)


,asin,cat_2,cat_3,cat_4_clean,image_url
0,0001487795,Kitchen & Dining,Dining & Entertaining,Dinnerware,NaN
1,0002020300,Home Dcor,Candles & Holders,Candles,NaN
2,0006564224,Kitchen & Dining,Dining & Entertaining,Glassware & Drinkware,NaN


## 5. Co-purchase pairs — the categories on both ends

One row per edge: the source product with its categories and image, the
`also_buy` target with its own. `~5M rows` at this grain, so the six category
columns and the two image urls are cast to `category` dtype — the categories
hold a few hundred distinct values between them, and while the urls run to
about a million, one asin is repeated across many edges, so the codes-plus-
dictionary layout still beats storing the string on every row.

`Not in catalogue` marks a target that appears in neither table. Those are real
edges — the co-purchase happened — pointing at books, electronics and anything
else Amazon sells, and the share of them is worth reading off the coverage
count below before aggregating. A null `dst_image_url` is a weaker statement
than that: the item may be in the catalogue and simply have no image listed,
which is the majority case on both ends.

In [56]:
NOT_IN_CATALOGUE = "Not in catalogue"

pairs = (
    df_base.loc[df_base["also_buy_n"] > 0,
                ["asin", "cat_2", "cat_3", "cat_4_clean", "image_url", "also_buy"]]
    .explode("also_buy", ignore_index=True)
    .rename(columns={
        "asin": "src_asin",
        "cat_2": "src_cat_2",
        "cat_3": "src_cat_3",
        "cat_4_clean": "src_cat_4",
        "image_url": "src_image_url",
        "also_buy": "dst_asin",
    })
)

pairs = pairs.merge(
    cat_lookup.rename(columns={
        "asin": "dst_asin",
        "cat_2": "dst_cat_2",
        "cat_3": "dst_cat_3",
        "cat_4_clean": "dst_cat_4",
        "image_url": "dst_image_url",
    }),
    on="dst_asin",
    how="left",
)

resolved = pairs["dst_cat_2"].notna()
cat_cols = ["src_cat_2", "src_cat_3", "src_cat_4", "dst_cat_2", "dst_cat_3", "dst_cat_4"]
img_cols = ["src_image_url", "dst_image_url"]
for col in ["dst_cat_2", "dst_cat_3", "dst_cat_4"]:
    pairs[col] = pairs[col].fillna(NOT_IN_CATALOGUE)
pairs[cat_cols + img_cols] = pairs[cat_cols + img_cols].astype("category")

pairs = pairs[["src_asin"] + cat_cols[:3] + ["src_image_url",
               "dst_asin"] + cat_cols[3:] + ["dst_image_url"]]

with_src_img = pairs["src_image_url"].notna()
with_dst_img = pairs["dst_image_url"].notna()
print(f"co-purchase pairs   : {len(pairs):,}")
print(f"distinct source asins: {pairs['src_asin'].nunique():,}")
print(f"distinct target asins: {pairs['dst_asin'].nunique():,}")
print(f"targets resolved     : {resolved.sum():,} ({resolved.mean():.1%})")
print(f"'{NOT_IN_CATALOGUE}'  : {(~resolved).sum():,} ({(~resolved).mean():.1%})")
print(f"src image url        : {with_src_img.sum():,} ({with_src_img.mean():.1%})")
print(f"dst image url        : {with_dst_img.sum():,} ({with_dst_img.mean():.1%})")
print(f"both ends imaged     : {(with_src_img & with_dst_img).sum():,} "
      f"({(with_src_img & with_dst_img).mean():.1%})")
print(f"memory               : {pairs.memory_usage(deep=True).sum() / 1e9:.2f} GB")

pairs.head(10)

co-purchase pairs   : 3,098,540
distinct source asins: 154,769
distinct target asins: 660,127
targets resolved     : 1,026,873 (33.1%)
'Not in catalogue'  : 2,071,667 (66.9%)
src image url        : 2,106,353 (68.0%)
dst image url        : 668,159 (21.6%)
both ends imaged     : 496,516 (16.0%)
memory               : 0.44 GB


,src_asin,src_cat_2,src_cat_3,src_cat_4,src_image_url,dst_asin,dst_cat_2,dst_cat_3,dst_cat_4,dst_image_url
0,0001487795,Kitchen & Dining,Dining & Entertaining,Dinnerware,NaN,B0001XR2F2,Kitchen & Dining,Dining & Entertaining,Dinnerware,https://images-na.ssl-images-amazon.com/images...
1,0001487795,Kitchen & Dining,Dining & Entertaining,Dinnerware,NaN,B01LY51HUN,Not in catalogue,Not in catalogue,Not in catalogue,NaN
2,0001487795,Kitchen & Dining,Dining & Entertaining,Dinnerware,NaN,B07CXZ9C5B,Not in catalogue,Not in catalogue,Not in catalogue,NaN
3,0001487795,Kitchen & Dining,Dining & Entertaining,Dinnerware,NaN,0310258952,Not in catalogue,Not in catalogue,Not in catalogue,NaN
4,0439903491,Wall Art,Posters & Prints,Missing,https://images-na.ssl-images-amazon.com/images...,0439900581,Not in catalogue,Not in catalogue,Not in catalogue,NaN
5,0456680012,Bedding,Kids' Bedding,Duvet Covers & Sets,https://images-na.ssl-images-amazon.com/images...,B00I8TCB02,Not in catalogue,Not in catalogue,Not in catalogue,NaN
6,0456680012,Bedding,Kids' Bedding,Duvet Covers & Sets,https://images-na.ssl-images-amazon.com/images...,B00K5B0PCC,Bedding,Kids' Bedding,Sheets & Pillowcases,https://images-na.ssl-images-amazon.com/images...
7,0456680012,Bedding,Kids' Bedding,Duvet Covers & Sets,https://images-na.ssl-images-amazon.com/images...,B00P8BOYNU,Not in catalogue,Not in catalogue,Not in catalogue,NaN
8,0456680012,Bedding,Kids' Bedding,Duvet Covers & Sets,https://images-na.ssl-images-amazon.com/images...,B01AB1CFA0,Bedding,Kids' Bedding,Comforters & Sets,https://images-na.ssl-images-amazon.com/images...
9,0456680012,Bedding,Kids' Bedding,Duvet Covers & Sets,https://images-na.ssl-images-amazon.com/images...,B01KTXJFYY,Not in catalogue,Not in catalogue,Not in catalogue,NaN


In [57]:
in_catalogue = pairs[pairs["dst_cat_2"] != NOT_IN_CATALOGUE][['src_cat_2', 'src_cat_3', 'src_cat_4', 'dst_cat_2', 'dst_cat_3', 'dst_cat_4']].drop_duplicates()

print(f"resolved edges     : {(pairs['dst_cat_2'] != NOT_IN_CATALOGUE).sum():,}")
print(f"distinct rows      : {len(in_catalogue):,}")
in_catalogue

resolved edges     : 1,026,873
distinct rows      : 30,392


,src_cat_2,src_cat_3,src_cat_4,dst_cat_2,dst_cat_3,dst_cat_4
0,Kitchen & Dining,Dining & Entertaining,Dinnerware,Kitchen & Dining,Dining & Entertaining,Dinnerware
6,Bedding,Kids' Bedding,Duvet Covers & Sets,Bedding,Kids' Bedding,Sheets & Pillowcases
8,Bedding,Kids' Bedding,Duvet Covers & Sets,Bedding,Kids' Bedding,Comforters & Sets
11,Bedding,Kids' Bedding,Duvet Covers & Sets,Storage & Organization,Laundry Storage & Organization,Laundry Storage & Organization_Other
14,Bedding,Kids' Bedding,Duvet Covers & Sets,Home Dcor,Kids' Room Dcor,Window Treatments
...,...,...,...,...,...,...
3097409,Kitchen & Dining,Storage & Organization,Travel & To-Go Food Containers,Bedding,Kids' Bedding,Comforters & Sets
3097524,Bedding,Bed Pillows & Positioners,Missing,Kitchen & Dining,Kitchen Utensils & Gadgets,Salt & Pepper
3097733,Home Dcor,Vases,Missing,Kitchen & Dining,Bakeware,Decorating Tools
3098119,Bedding,"Decorative Pillows, Inserts & Covers",Throw Pillow Covers,Home Dcor,Window Stickers & Films,Missing


## 6. Which of the mapped pairs does the data actually show?

`complementary_category_map.csv` is the hand-built answer to the question this
notebook asks of the data: 2,422 rules of the form `source cat_2/3/4 -> target
cat_2/3/4`, each with a `confidence`. `in_catalogue` is the observed answer —
the distinct category pairs that real `also_buy` edges trace out. One question
here: **for each rule in the map, is that pair present in `in_catalogue`?**

Source and target are the two ends of a co-purchase edge and come from
different tables. The source is the anchor item, from `df_features.pkl`, so it
is always one of the 69 schema categories; the target is an asin out of that
item's `also_buy`, looked up in the filtered catalogue CSV, which was never
narrowed to those 69. The map reads the same way — `source_cat*` is the anchor,
`target_cat*` the thing bought alongside it — so the two tables line up
directly.

The headline number is the strict one: **all six columns equal at once** —
`src_cat_2 == source_cat2`, `src_cat_3 == source_cat3`, `src_cat_4 ==
source_cat4`, and the same three on the target side — over the 1,843 rules
that actually name both leaves. The 579 rules leaving a leaf blank cannot be
tested that way and are excluded from that denominator rather than counted as
hits or misses.

The looser readings follow it, for the rules the strict test cannot judge:

- **cat_3 grain** — is the branch pair `(source cat_2/cat_3) -> (target
  cat_2/cat_3)` present at all? This is the loose reading, ignoring leaves.
- **cat_4 grain** — the full pair, leaves included. 404 rules leave
  `source_cat4` blank and 207 leave `target_cat4` blank; those are stated at
  the branch grain, so any observed leaf under them satisfies the rule rather
  than failing it.

Every category the map names is in `category_taxonomy.json`, so its leaves are
directly comparable with `cat_4_clean`. The one asymmetry to keep in mind: an
observed item whose leaf failed review folded to `<cat_3>_Other` and can no
longer match the leaf it was named at, so cat_4 coverage is a floor. That
folding is small on the source side (~1% of rows) but reaches ~11% on the
target side.

The miss breakdown says which of three things went wrong for every rule that
is not confirmed: the branch pair was never seen, one end's leaf was never
seen under that branch, or both leaves were seen but never on the same edge.

In [58]:
cat_map = pd.read_csv(CATEGORY_MAP_PATH)
print(f"map rules: {len(cat_map):,} | blank source_cat4: "
      f"{cat_map['source_cat4'].isna().sum():,} | blank target_cat4: "
      f"{cat_map['target_cat4'].isna().sum():,}")

MAP_KEYS_3 = ["source_cat2", "source_cat3", "target_cat2", "target_cat3"]
OBS_KEYS_3 = ["src_cat_2", "src_cat_3", "dst_cat_2", "dst_cat_3"]
MAP_KEYS_4 = ["source_cat2", "source_cat3", "source_cat4",
              "target_cat2", "target_cat3", "target_cat4"]
OBS_KEYS_4 = ["src_cat_2", "src_cat_3", "src_cat_4",
              "dst_cat_2", "dst_cat_3", "dst_cat_4"]

# Plain strings: in_catalogue carries `category` dtype, which does not merge
# against the map's object columns.
obs = in_catalogue[OBS_KEYS_4].astype(str)

# THE MATCH: all six columns equal at once — src_cat_2 == source_cat2, ...,
# dst_cat_4 == target_cat4 — over the rules that actually pin down both
# leaves. No wildcards, no branch-level fallback, no partial credit.
strict_rules = cat_map.dropna(subset=["source_cat4", "target_cat4"])
obs_pairs = obs[OBS_KEYS_4].drop_duplicates()
strict_match = (
    strict_rules[MAP_KEYS_4]
    .merge(obs_pairs, left_on=MAP_KEYS_4, right_on=OBS_KEYS_4,
           how="left", indicator=True)["_merge"].eq("both").to_numpy()
)
print(f"\nexact item -> complement pairs, both cat_4 present in the map:")
print(f"  {strict_match.sum():,}/{len(strict_rules):,} ({strict_match.mean():.1%}) "
      f"also in in_catalogue")
print(f"  (map: {len(strict_rules):,} of {len(cat_map):,} rules name both leaves | "
      f"in_catalogue: {len(obs_pairs):,} distinct pairs)")

# cat_3 grain — is the branch pair traced by any co-purchase edge at all?
covered_3 = (
    cat_map[MAP_KEYS_3]
    .merge(obs[OBS_KEYS_3].drop_duplicates(),
           left_on=MAP_KEYS_3, right_on=OBS_KEYS_3, how="left", indicator=True)
    ["_merge"].eq("both").to_numpy()
)

# cat_4 grain — every observed pair sharing the rule's branches, scored on
# whether the named leaves match. A blank leaf in the map is a wildcard, so it
# is satisfied by whichever leaf was observed.
hits = (
    cat_map[MAP_KEYS_4].reset_index()
    .merge(obs, left_on=MAP_KEYS_3, right_on=OBS_KEYS_3, how="inner")
)
hits["src_ok"] = hits["source_cat4"].isna() | hits["source_cat4"].eq(hits["src_cat_4"])
hits["tgt_ok"] = hits["target_cat4"].isna() | hits["target_cat4"].eq(hits["dst_cat_4"])
hits["both_ok"] = hits["src_ok"] & hits["tgt_ok"]

rule_hits = (
    hits.groupby("index")[["src_ok", "tgt_ok", "both_ok"]].any()
    .reindex(cat_map.index, fill_value=False).astype(bool)
)
covered_4 = rule_hits["both_ok"].to_numpy()

cat_map_coverage = cat_map.assign(covered_cat_3=covered_3, covered_cat_4=covered_4)
cat_map_coverage["exact_match"] = pd.Series(strict_match, index=strict_rules.index)

print(f"\nlooser readings, all {len(cat_map):,} rules (blank leaf = wildcard)")
print(f"  cat_3 grain (branch pair): {covered_3.sum():,}/{len(cat_map):,} "
      f"({covered_3.mean():.1%})")
print(f"  cat_4 grain (full pair)  : {covered_4.sum():,}/{len(cat_map):,} "
      f"({covered_4.mean():.1%})")

# Why the rest miss.
branch_seen = cat_map.index.isin(hits["index"].unique())
missed = rule_hits[branch_seen & ~covered_4]
print(f"\nof the {(~covered_4).sum():,} rules not confirmed at cat_4 grain")
print(f"  branch pair never observed      : {(~branch_seen).sum():,}")
print(f"  source leaf never seen there    : {(~missed['src_ok']).sum():,}")
print(f"  target leaf never seen there    : {(~missed['tgt_ok']).sum():,}")
print(f"  both leaves seen, never together: "
      f"{(missed['src_ok'] & missed['tgt_ok']).sum():,}")

# The same question asked of the categories the map names, not its rules.
print()
for label, map_cols, obs_cols in [
    ("source categories, cat_3 grain", MAP_KEYS_3[:2], OBS_KEYS_3[:2]),
    ("source categories, cat_4 grain", MAP_KEYS_4[:3], OBS_KEYS_4[:3]),
    ("target categories, cat_3 grain", MAP_KEYS_3[2:], OBS_KEYS_3[2:]),
    ("target categories, cat_4 grain", MAP_KEYS_4[3:], OBS_KEYS_4[3:]),
]:
    named = cat_map[map_cols].dropna().drop_duplicates()
    seen = obs[obs_cols].drop_duplicates()
    n_hit = len(named.merge(seen, left_on=map_cols, right_on=obs_cols, how="inner"))
    print(f"{label}: {n_hit:,}/{len(named):,} ({n_hit / len(named):.1%})")

# Do the rules the curator was surest of show up more often?
bands = pd.cut(cat_map_coverage["confidence"], [0.4, 0.6, 0.7, 0.8, 1.0])
band_coverage = (
    cat_map_coverage.groupby(bands, observed=True)
    .agg(rules=("confidence", "size"),
         cat_3=("covered_cat_3", "mean"),
         cat_4=("covered_cat_4", "mean"))
)
print("\ncoverage by confidence band:")
print(band_coverage.to_string(
    formatters={"cat_3": "{:.1%}".format, "cat_4": "{:.1%}".format}))

# The gap worth reading: mapped pairs with no co-purchase edge behind them.
cat_map_coverage.loc[~cat_map_coverage["covered_cat_4"],
                     MAP_KEYS_4 + ["confidence", "covered_cat_3"]].head(20)

map rules: 2,422 | blank source_cat4: 404 | blank target_cat4: 207

exact item -> complement pairs, both cat_4 present in the map:
  924/1,843 (50.1%) also in in_catalogue
  (map: 1,843 of 2,422 rules name both leaves | in_catalogue: 30,392 distinct pairs)

looser readings, all 2,422 rules (blank leaf = wildcard)
  cat_3 grain (branch pair): 2,351/2,422 (97.1%)
  cat_4 grain (full pair)  : 1,322/2,422 (54.6%)

of the 1,100 rules not confirmed at cat_4 grain
  branch pair never observed      : 71
  source leaf never seen there    : 523
  target leaf never seen there    : 225
  both leaves seen, never together: 368

source categories, cat_3 grain: 69/69 (100.0%)
source categories, cat_4 grain: 359/361 (99.4%)
target categories, cat_3 grain: 54/54 (100.0%)
target categories, cat_4 grain: 145/145 (100.0%)

coverage by confidence band:
            rules  cat_3 cat_4
confidence                    
(0.4, 0.6]    501  94.6% 47.9%
(0.6, 0.7]    810  95.9% 52.3%
(0.7, 0.8]    573  98.1% 52.7%
(0

,source_cat2,source_cat3,source_cat4,target_cat2,target_cat3,target_cat4,confidence,covered_cat_3
9,Bath,Bathroom Accessories,Bathroom Accessory Sets,Bath,Towels,Beach Towels,0.81,True
10,Bath,Bathroom Accessories,Bathroom Accessory Sets,Furniture,Bathroom Furniture,NaN,0.70,True
11,Bath,Bathroom Accessories,Bathroom Accessory Sets,Home Dcor,Mirrors,NaN,0.60,True
12,Bath,Bathroom Accessories,Bathroom Mirrors,Bath,Bath Rugs,NaN,0.90,True
13,Bath,Bathroom Accessories,Bathroom Mirrors,Bath,Towels,Towel Sets,0.81,True
14,Bath,Bathroom Accessories,Bathroom Mirrors,Bath,Towels,Beach Towels,0.81,True
15,Bath,Bathroom Accessories,Bathroom Mirrors,Bath,Towels,Bath Towels,0.81,True
16,Bath,Bathroom Accessories,Bathroom Mirrors,Furniture,Bathroom Furniture,NaN,0.70,True
21,Bath,Bathroom Accessories,Bathtub Accessories,Bath,Towels,Beach Towels,0.81,True
22,Bath,Bathroom Accessories,Bathtub Accessories,Furniture,Bathroom Furniture,NaN,0.70,True


In [59]:
cat_map.head()

,source_cat2,source_cat3,source_cat4,target_cat2,target_cat3,target_cat4,confidence
0,Bath,Bath Rugs,NaN,Bath,Bathroom Accessories,"Shower Curtains, Hooks & Liners",0.81
1,Bath,Bath Rugs,NaN,Bath,Towels,Bath Towels,0.81
2,Bath,Bath Rugs,NaN,Bath,Towels,Beach Towels,0.81
3,Bath,Bath Rugs,NaN,Bath,Towels,Towel Sets,0.81
4,Bath,Bath Rugs,NaN,Bath,Bathroom Accessories,Bathtub Accessories,0.81


In [60]:
# in_catalogue.shape
in_catalogue[in_catalogue['src_cat_4'].isna() == False]

,src_cat_2,src_cat_3,src_cat_4,dst_cat_2,dst_cat_3,dst_cat_4
0,Kitchen & Dining,Dining & Entertaining,Dinnerware,Kitchen & Dining,Dining & Entertaining,Dinnerware
6,Bedding,Kids' Bedding,Duvet Covers & Sets,Bedding,Kids' Bedding,Sheets & Pillowcases
8,Bedding,Kids' Bedding,Duvet Covers & Sets,Bedding,Kids' Bedding,Comforters & Sets
11,Bedding,Kids' Bedding,Duvet Covers & Sets,Storage & Organization,Laundry Storage & Organization,Laundry Storage & Organization_Other
14,Bedding,Kids' Bedding,Duvet Covers & Sets,Home Dcor,Kids' Room Dcor,Window Treatments
...,...,...,...,...,...,...
3097409,Kitchen & Dining,Storage & Organization,Travel & To-Go Food Containers,Bedding,Kids' Bedding,Comforters & Sets
3097524,Bedding,Bed Pillows & Positioners,Missing,Kitchen & Dining,Kitchen Utensils & Gadgets,Salt & Pepper
3097733,Home Dcor,Vases,Missing,Kitchen & Dining,Bakeware,Decorating Tools
3098119,Bedding,"Decorative Pillows, Inserts & Covers",Throw Pillow Covers,Home Dcor,Window Stickers & Films,Missing


In [61]:
cat_map.head()

,source_cat2,source_cat3,source_cat4,target_cat2,target_cat3,target_cat4,confidence
0,Bath,Bath Rugs,NaN,Bath,Bathroom Accessories,"Shower Curtains, Hooks & Liners",0.81
1,Bath,Bath Rugs,NaN,Bath,Towels,Bath Towels,0.81
2,Bath,Bath Rugs,NaN,Bath,Towels,Beach Towels,0.81
3,Bath,Bath Rugs,NaN,Bath,Towels,Towel Sets,0.81
4,Bath,Bath Rugs,NaN,Bath,Bathroom Accessories,Bathtub Accessories,0.81


In [85]:
print(in_catalogue.shape)
print(cat_map.shape)

(30392, 6)
(2422, 7)


## 7. The match funnel

`complementary_category_map.csv` is the source here, every rule in it is 100%,
and each rule falls into exactly one bucket. A rule counts as **matched** only
when all six columns line up against a row of `in_catalogue` at once —
`src_cat_2 == source_cat2`, `src_cat_3 == source_cat3`, `src_cat_4 ==
source_cat4`, and the same three on the target side.

A quarter of the map cannot reach that test: a rule with a blank `source_cat4`
or `target_cat4` has no six-column form to compare, so it is set aside rather
than scored as a miss. The rules that *are* comparable then split into the
exact matches and the ones the co-purchase data never produced, and those are
sorted by how far off they were — whether the branch pair itself was never
seen, one end's leaf was never seen under that branch, or both leaves were
seen separately but never on the same edge.

In [62]:
funnel_map = pd.read_csv(CATEGORY_MAP_PATH)

MAP_COLS = ["source_cat2", "source_cat3", "source_cat4",
            "target_cat2", "target_cat3", "target_cat4"]
OBS_COLS = ["src_cat_2", "src_cat_3", "src_cat_4",
            "dst_cat_2", "dst_cat_3", "dst_cat_4"]
BRANCH_MAP = MAP_COLS[:2] + MAP_COLS[3:5]
BRANCH_OBS = OBS_COLS[:2] + OBS_COLS[3:5]

observed = in_catalogue[OBS_COLS].astype(str)
observed_pairs = observed.drop_duplicates()
status = pd.Series("matched", index=funnel_map.index, dtype=object)

# 1. Rules that cannot reach a six-column test at all.
src_blank = funnel_map["source_cat4"].isna()
tgt_blank = funnel_map["target_cat4"].isna()
status[src_blank & ~tgt_blank] = "no source_cat4"
status[~src_blank & tgt_blank] = "no target_cat4"
status[src_blank & tgt_blank] = "no cat_4 on either end"

# 2. Of the rest, the exact match: all six columns at once.
comparable = ~(src_blank | tgt_blank)
matched = pd.Series(False, index=funnel_map.index)
matched[comparable] = (
    funnel_map.loc[comparable, MAP_COLS]
    .merge(observed_pairs, left_on=MAP_COLS, right_on=OBS_COLS,
           how="left", indicator=True)["_merge"].eq("both").to_numpy()
)

# 3. For the comparable rules that missed: how close did they get? Every
#    observed pair sharing the rule's two branches is a candidate, so a rule
#    is graded on whether its leaves ever showed up there at all.
unmatched = comparable & ~matched
near = (
    funnel_map.loc[unmatched, MAP_COLS].reset_index()
    .merge(observed, left_on=BRANCH_MAP, right_on=BRANCH_OBS, how="inner")
)
branch_seen = pd.Series(funnel_map.index.isin(near["index"].unique()),
                        index=funnel_map.index)
src_seen = (near.assign(k=near["source_cat4"].eq(near["src_cat_4"]))
            .groupby("index")["k"].any().reindex(funnel_map.index, fill_value=False))
tgt_seen = (near.assign(k=near["target_cat4"].eq(near["dst_cat_4"]))
            .groupby("index")["k"].any().reindex(funnel_map.index, fill_value=False))

status[unmatched & ~branch_seen] = "branch pair never observed"
status[unmatched & branch_seen & ~src_seen] = "source cat_4 never seen in branch"
status[unmatched & branch_seen & src_seen & ~tgt_seen] = "target cat_4 never seen in branch"
status[unmatched & branch_seen & src_seen & tgt_seen] = "both leaves seen, never paired"

cat_map_matched = funnel_map.assign(match_status=status)

total = len(funnel_map)


def line(label, mask, depth=0):
    n = int(mask.sum())
    print(f"{'  ' * depth}{label:<{46 - 2 * depth}} {n:>6,}  {n / total:>6.1%}")


print(f"{'complementary_category_map.csv':<46} {total:>6,}  100.0%")
line("not comparable - a cat_4 is missing", src_blank | tgt_blank, 1)
line("source_cat4 blank", src_blank & ~tgt_blank, 2)
line("target_cat4 blank", ~src_blank & tgt_blank, 2)
line("both blank", src_blank & tgt_blank, 2)
line("comparable - both cat_4 present", comparable, 1)
line("MATCHED - all six columns", matched, 2)
line("not matched", unmatched, 2)
line("branch pair never observed", unmatched & ~branch_seen, 3)
line("source cat_4 never seen in branch", unmatched & branch_seen & ~src_seen, 3)
line("target cat_4 never seen in branch",
     unmatched & branch_seen & src_seen & ~tgt_seen, 3)
line("both leaves seen, never paired",
     unmatched & branch_seen & src_seen & tgt_seen, 3)

print(f"\nmatched, as a share of the comparable rules only: "
      f"{matched.sum():,}/{comparable.sum():,} "
      f"({matched.sum() / comparable.sum():.1%})")

match_funnel = (
    cat_map_matched.groupby("match_status").size().rename("rules").to_frame()
    .assign(share=lambda d: d["rules"] / total)
    .sort_values("rules", ascending=False)
)
match_funnel

complementary_category_map.csv                  2,422  100.0%
  not comparable - a cat_4 is missing             579   23.9%
    source_cat4 blank                             372   15.4%
    target_cat4 blank                             175    7.2%
    both blank                                     32    1.3%
  comparable - both cat_4 present               1,843   76.1%
    MATCHED - all six columns                     924   38.2%
    not matched                                   919   37.9%
      branch pair never observed                   31    1.3%
      source cat_4 never seen in branch           451   18.6%
      target cat_4 never seen in branch            69    2.8%
      both leaves seen, never paired              368   15.2%

matched, as a share of the comparable rules only: 924/1,843 (50.1%)


,rules,share
match_status,,
matched,924,0.38
source cat_4 never seen in branch,451,0.19
no source_cat4,372,0.15
"both leaves seen, never paired",368,0.15
no target_cat4,175,0.07
target cat_4 never seen in branch,69,0.03
no cat_4 on either end,32,0.01
branch pair never observed,31,0.01


## 8. Where the match is strong and where it is not

The comparison is a left join: the map on the left, the distinct pairs of
`in_catalogue` on the right, joined on all six columns at once. Because the
right side is deduplicated on exactly those six columns, the join returns one
row per rule and nothing fans out — so `_merge == "both"` is a per-rule verdict
and the flag can be grouped by any category column.

The 50.1% headline is an average over branches that behave very differently.
Grouping it by `cat_2`, `cat_3` and `cat_4` on either end shows which parts of
the map the co-purchase data backs up and which parts it does not.

In [63]:
# The left join, one row per rule: the map on the left, in_catalogue's distinct
# six-column pairs on the right. The right side is unique on the join keys, so
# no rule fans out into several rows.
map_vs_observed = funnel_map.merge(
    observed_pairs, left_on=MAP_COLS, right_on=OBS_COLS, how="left", indicator=True
)
map_vs_observed["matched"] = map_vs_observed["_merge"].eq("both")
assert len(map_vs_observed) == len(funnel_map)

# Only rules naming both leaves can match; the rest would drag every rate down
# for a reason that has nothing to do with the category.
comparable_rules = map_vs_observed[
    map_vs_observed["source_cat4"].notna() & map_vs_observed["target_cat4"].notna()
]
print(f"comparable rules: {len(comparable_rules):,} | matched: "
      f"{comparable_rules['matched'].sum():,} "
      f"({comparable_rules['matched'].mean():.1%})")


def match_rate_by(col, min_rules=1):
    """Match rate per value of `col`, worst last, thin groups dropped."""
    out = (comparable_rules.groupby(col)
           .agg(rules=("matched", "size"), matched=("matched", "sum"))
           .assign(rate=lambda d: d["matched"] / d["rules"]))
    return out[out["rules"] >= min_rules].sort_values("rate", ascending=False)


def show(col, min_rules=1, head=None):
    out = match_rate_by(col, min_rules)
    print(f"\n== {col} ==" + (f"  ({len(out)} values, top/bottom {head})" if head else ""))
    fmt = {"rate": "{:.1%}".format}
    if head is None or len(out) <= 2 * head:
        print(out.to_string(formatters=fmt))
    else:
        print(out.head(head).to_string(formatters=fmt))
        print("...")
        print(out.tail(head).to_string(formatters=fmt, header=False))


show("source_cat2")
show("target_cat2")
show("source_cat3", min_rules=10, head=8)
show("target_cat3", min_rules=10, head=8)
# cat_4 spreads 1,843 rules over ~360 source leaves, so the threshold has to
# drop or nothing survives it.
show("source_cat4", min_rules=5, head=8)
show("target_cat4", min_rules=5, head=8)

# Every pairing of the two top levels, as a grid. Rates over a handful of
# rules read as 0% or 100% and mean little, so the counts go next to them.
print("\n== source_cat2 x target_cat2: match rate (rules) ==")
grid = comparable_rules.pivot_table(index="source_cat2", columns="target_cat2",
                                    values="matched", aggfunc=["mean", "size"])
cells = grid["mean"].map(lambda v: "" if pd.isna(v) else f"{v:.0%}")
counts = grid["size"].map(lambda v: "" if pd.isna(v) else f" ({v:,.0f})")
print((cells + counts).replace("", "-").to_string())

match_rate_by("source_cat3", min_rules=10)

comparable rules: 1,843 | matched: 924 (50.1%)

== source_cat2 ==
                  rules  matched  rate
source_cat2                           
Kitchen & Dining    829      556 67.1%
Bath                 56       29 51.8%
Home Dcor           517      204 39.5%
Bedding             173       63 36.4%
Furniture           268       72 26.9%

== target_cat2 ==
                  rules  matched  rate
target_cat2                           
Kitchen & Dining    865      571 66.0%
Bath                 56       29 51.8%
Home Dcor           614      226 36.8%
Bedding             250       83 33.2%
Furniture            58       15 25.9%

== source_cat3 ==  (39 values, top/bottom 8)
                             rules  matched  rate
source_cat3                                      
Kitchen Utensils & Gadgets     114      100 87.7%
Dining & Entertaining           24       21 87.5%
Storage & Organization          48       39 81.2%
Bakeware                        88       71 80.7%
Cutlery & Knife Accesso

,rules,matched,rate
source_cat3,,,
Kitchen Utensils & Gadgets,114,100,0.88
Dining & Entertaining,24,21,0.88
Storage & Organization,48,39,0.81
Bakeware,88,71,0.81
Cutlery & Knife Accessories,96,77,0.80
Artificial Plants & Flowers,24,19,0.79
Kids' Room Dcor,25,19,0.76
Candles & Holders,29,22,0.76
Towels,20,15,0.75


## 9. Checking a category by hand

The left join itself is `map_vs_observed`: one row per map rule, `matched`
saying whether all six columns landed on a row of `in_catalogue`. It answers
*whether* a rule matched but not *what the data had instead* — joining on all
six columns means the observed columns only ever echo the map's own values.

`compare()` below puts the two sides next to each other for whatever slice you
name. It takes any map column as a filter — `compare(source_cat3="Slipcovers")`
— and prints the map's rules for that slice against the co-purchase pairs
observed under the same branches, carrying the edge count so a pair backed by
40,000 co-purchases is not read the same as one backed by two.

In [64]:
# Edge counts, so manual checks can weigh a pair by how much traffic is behind
# it. in_catalogue is deduplicated, so the counts come from `pairs` itself.
observed_counts = (
    pairs.loc[pairs["dst_cat_2"] != NOT_IN_CATALOGUE, OBS_COLS].astype(str)
    .groupby(OBS_COLS, observed=True).size().rename("edges")
    .reset_index().sort_values("edges", ascending=False)
)
MAP_TO_OBS = dict(zip(MAP_COLS, OBS_COLS))
print(f"observed_counts: {len(observed_counts):,} distinct pairs, "
      f"{observed_counts['edges'].sum():,} edges")


def compare(top=12, **filters):
    """The map's rules for a slice, next to what the data actually shows there.

    Filter on any map column: compare(source_cat3="Slipcovers"),
    compare(source_cat2="Furniture", target_cat3="Area Rugs, Runners & Pads").
    """
    rules, seen = map_vs_observed, observed_counts
    for col, val in filters.items():
        rules = rules[rules[col] == val]
        seen = seen[seen[MAP_TO_OBS[col]] == val]
    if not len(rules):
        print(f"no map rules for {filters}")
        return seen.head(top)

    testable = rules["source_cat4"].notna() & rules["target_cat4"].notna()
    print(f"{filters}")
    print(f"  map rules {len(rules):,} | comparable {testable.sum():,} | "
          f"matched {rules['matched'].sum():,}"
          + (f" ({rules.loc[testable, 'matched'].mean():.1%} of comparable)"
             if testable.any() else ""))
    print(f"  observed pairs under these branches: {len(seen):,} "
          f"({seen['edges'].sum():,} edges)")

    print(f"\n-- the map's rules (top {top} by confidence) --")
    print(rules.sort_values("confidence", ascending=False)
          .head(top)[MAP_COLS + ["confidence", "matched"]].to_string(index=False))
    print(f"\n-- what the co-purchase data shows (top {top} by edges) --")
    print(seen.head(top).to_string(index=False))
    return rules


# A branch the map calls right most of the time, and one it mostly misses.
compare(source_cat3="Kitchen Utensils & Gadgets", top=6)
print("\n" + "=" * 100 + "\n")
compare(source_cat3="Slipcovers", top=6)

observed_counts: 30,392 distinct pairs, 1,026,873 edges
{'source_cat3': 'Kitchen Utensils & Gadgets'}
  map rules 120 | comparable 114 | matched 100 (87.7% of comparable)
  observed pairs under these branches: 3,129 (118,130 edges)

-- the map's rules (top 6 by confidence) --
     source_cat2                source_cat3            source_cat4      target_cat2 target_cat3                  target_cat4  confidence  matched
Kitchen & Dining Kitchen Utensils & Gadgets       Bar & Wine Tools Kitchen & Dining    Cookware                Cookware Sets        0.81     True
Kitchen & Dining Kitchen Utensils & Gadgets            Jar Openers Kitchen & Dining    Cookware                Cookware Sets        0.81    False
Kitchen & Dining Kitchen Utensils & Gadgets Salad Tools & Spinners Kitchen & Dining    Cookware                     All Pans        0.81     True
Kitchen & Dining Kitchen Utensils & Gadgets    Pasta & Pizza Tools Kitchen & Dining    Cookware Steamers, Stock & Pasta Pots        0.81   

,source_cat2,source_cat3,source_cat4,target_cat2,target_cat3,target_cat4,confidence,src_cat_2,src_cat_3,src_cat_4,dst_cat_2,dst_cat_3,dst_cat_4,_merge,matched
1311,Home Dcor,Slipcovers,Armchair Slipcovers,Bedding,"Decorative Pillows, Inserts & Covers",Throw Pillow Covers,0.72,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False
1312,Home Dcor,Slipcovers,Armchair Slipcovers,Bedding,"Decorative Pillows, Inserts & Covers",Throw Pillows,0.72,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False
1313,Home Dcor,Slipcovers,Armchair Slipcovers,Bedding,"Decorative Pillows, Inserts & Covers",Pillow Inserts,0.72,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False
1314,Home Dcor,Slipcovers,Armchair Slipcovers,Bedding,Blankets & Throws,Throws,0.63,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False
1315,Home Dcor,Slipcovers,Armchair Slipcovers,Bedding,Blankets & Throws,Bed Blankets,0.63,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False
1316,Home Dcor,Slipcovers,Armchair Slipcovers,Bedding,Blankets & Throws,Wearable Blankets,0.63,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False
1317,Home Dcor,Slipcovers,Dining Chair Slipcovers,Bedding,"Decorative Pillows, Inserts & Covers",Throw Pillows,0.72,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False
1318,Home Dcor,Slipcovers,Dining Chair Slipcovers,Bedding,"Decorative Pillows, Inserts & Covers",Throw Pillow Covers,0.72,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False
1319,Home Dcor,Slipcovers,Dining Chair Slipcovers,Bedding,"Decorative Pillows, Inserts & Covers",Pillow Inserts,0.72,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False
1320,Home Dcor,Slipcovers,Dining Chair Slipcovers,Bedding,Blankets & Throws,Throws,0.63,NaN,NaN,NaN,NaN,NaN,NaN,left_only,False


In [83]:
# map_vs_observed.shape
map_vs_observed[(map_vs_observed['source_cat2'] == map_vs_observed['src_cat_2']) &
                (map_vs_observed['source_cat3'] != map_vs_observed['src_cat_3']) &
                (map_vs_observed['source_cat4'] != map_vs_observed['src_cat_4'])].shape

(0, 15)